<a href="https://colab.research.google.com/github/Nithyaag73/AI-Engineering-Journey/blob/main/Twitter_Sentimental_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install pandas scikit-learn

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

In [26]:
import pandas as pd

cols = ['polarity', 'id', 'date', 'query', 'user', 'text']

df = pd.read_csv(
    'training.1600000.processed.noemoticon.csv',
    encoding='latin-1',
    names=cols,
    engine='python',
    on_bad_lines='skip'
)

# Keep only required columns
df = df[['polarity', 'text']]

print("After loading:", df.shape)
print(df['polarity'].unique())

After loading: (1529409, 2)
['orkinglife Ideally' '0' '4']


In [27]:
df['polarity'] = pd.to_numeric(df['polarity'], errors='coerce')

In [28]:
df = df.dropna(subset=['polarity', 'text'])

In [29]:
df['polarity'] = df['polarity'].astype(int)

In [30]:
df = df[df['polarity'].isin([0, 4])]

In [31]:
df['polarity'] = df['polarity'].map({0: 0, 4: 1})

In [32]:
df['clean_text'] = df['text'].astype(str).str.lower()

In [33]:
print("Final shape:", df.shape)
print(df['polarity'].value_counts())

Final shape: (1529408, 3)
polarity
1    800000
0    729408
Name: count, dtype: int64


In [34]:
from sklearn.model_selection import train_test_split

X = df['clean_text']
y = df['polarity']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 1223526
Test size: 305882


In [35]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF shape:", X_train_tfidf.shape)

TF-IDF shape: (1223526, 5000)


In [36]:
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score

bnb = BernoulliNB()
bnb.fit(X_train_tfidf, y_train)

bnb_pred = bnb.predict(X_test_tfidf)

print("Naive Bayes Accuracy:", accuracy_score(y_test, bnb_pred))

Naive Bayes Accuracy: 0.7648308824971721


In [37]:
from sklearn.svm import LinearSVC

svm = LinearSVC(max_iter=1000)
svm.fit(X_train_tfidf, y_train)

svm_pred = svm.predict(X_test_tfidf)

print("SVM Accuracy:", accuracy_score(y_test, svm_pred))

SVM Accuracy: 0.795221686794254


In [38]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(max_iter=100)
logreg.fit(X_train_tfidf, y_train)

logreg_pred = logreg.predict(X_test_tfidf)

print("Logistic Regression Accuracy:", accuracy_score(y_test, logreg_pred))

Logistic Regression Accuracy: 0.7959409183933673


In [39]:
print("\nMODEL COMPARISON")
print("Naive Bayes:", accuracy_score(y_test, bnb_pred))
print("SVM:", accuracy_score(y_test, svm_pred))
print("Logistic:", accuracy_score(y_test, logreg_pred))


MODEL COMPARISON
Naive Bayes: 0.7648308824971721
SVM: 0.795221686794254
Logistic: 0.7959409183933673


In [40]:
sample = [
    "I love this product",
    "This is the worst thing ever",
    "Not bad, okay experience"
]

sample_vec = vectorizer.transform(sample)

print("Naive Bayes:", bnb.predict(sample_vec))
print("SVM:", svm.predict(sample_vec))
print("Logistic:", logreg.predict(sample_vec))

Naive Bayes: [1 0 1]
SVM: [1 0 1]
Logistic: [1 0 1]
